# M03 — Transformación

[← Anterior](../M02-ingesta-preparacion/04-lab-calidad-limpieza.ipynb) · [Siguiente →](02-lab-enriquecimiento.ipynb)

Ya no estamos “leyendo el fichero”. Estamos **derivando columnas de negocio** a partir de las que ya tienes. La regla no es un `for` fila a fila: es una expresión que Spark aplica a toda la columna (`withColumn`).

Demo con cuatro líneas en memoria. No hace falta el staging: queremos ver el fallo de un descuento sucio *antes* de taparlo.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m03')
print(spark.version, spark.sparkContext.master)


## Una fórmula es una columna

GMV de línea = `qty * unit_price * (1 - discount)`. Si `discount` es `0.10`, quitas el 10 %. Si en el raw alguien escribió `1.50` (ciento cincuenta por ciento), `(1 - 1.50)` es negativo y el GMV **sale negativo**. No es un bug de Spark: es suciedad que la fórmula reproduce.

Al ejecutar verás cuatro filas. `O2` tiene `discount=1.50` y `gmv_line` negativo. Eso es lo que el lab de reglas tapa; ahora queremos **verlo**.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, when, lower, least, lit

lineas = spark.createDataFrame([
    Row(order_id="O1", qty=2, unit_price=10.0, discount=0.10, status="paid", channel="WEB"),
    Row(order_id="O2", qty=1, unit_price=80.0, discount=1.50, status="cancelled", channel="marketplace"),
    Row(order_id="O3", qty=3, unit_price=5.0, discount=0.0, status="paid", channel="app"),
    Row(order_id="O4", qty=1, unit_price=20.0, discount=0.0, status="pending", channel="store"),
])
# withColumn añade (o pisa) una columna. No hace falta un for.
crudo = lineas.withColumn(
    "gmv_line",
    col("qty") * col("unit_price") * (1 - col("discount")),
)
crudo.select("order_id", "discount", "gmv_line").show()  # O2 negativo


## Tres reglas encadenadas (y recalcular al final)

En el lab harás esto sobre el fact real. Aquí, sobre las 4 filas:

1. **Capar** el descuento a 1 (`least(discount, 1)`): no puedes descontar más del 100 %.
2. **Normalizar** el canal: `WEB`/`App`/`marketplace` no sirven para un `groupBy` limpio. Nos quedamos con `web`, `app`, `store` u `other`.
3. **Marcar** lo cobrable (`status == paid`). No filtres aún: el fact guarda todas las líneas; el flag decide en los KPIs.

Importante: si capas `discount` *después* de haber calculado `gmv_line` y no vuelves a calcular, el negativo **sigue**. Por eso el GMV se escribe **al final** de la cadena.

Al ejecutar: `O2` ya no tiene GMV negativo; `channel_norm` es `web` / `other` / `app` / `store`; `is_billable` es true solo en `paid`.


In [ ]:
fact = (
    crudo
    # 1) tope de descuento (no muta gmv_line todavía)
    .withColumn("discount", least(col("discount"), lit(1.0)))
    # 2) canal en minúsculas y set cerrado
    .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel")))
        .otherwise(lit("other")),
    )
    # 3) flag; el fact sigue teniendo las 4 filas
    .withColumn("is_billable", col("status") == "paid")
    # 4) ahora sí: GMV con el discount ya capado
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
)
fact.select(
    "order_id", "channel", "channel_norm", "discount", "gmv_line", "is_billable"
).show()


No uses `collect()` / `toPandas()` del fact entero. Si quieres mirar, `limit(20).toPandas()`.

**Siguiente:** [lab de enriquecimiento](02-lab-enriquecimiento.ipynb).
